# 01 — Engine API quickstart (mocked, no API key)

Walks through the engine's internals without touching any network. The LLM is mocked so every cell runs in seconds and the pipeline structure is visible.

**You will learn:**
1. How the 8-node graph is built and invoked.
2. What `State` looks like at each node boundary.
3. How `run_query` + `RunResult` wrap the raw graph output.
4. How a domain preset modifies the prompt.

**Time:** ~2 minutes. **Cost:** $0. **API key:** none.

## Step 1 — install

In [ ]:
%%capture
# Clone the repo and install engine + core/rag deps.
!git clone --depth 1 https://github.com/TheAiSingularity/agentic-research-engine-oss.git /content/engine-repo
!pip install -q langgraph openai rank_bm25 trafilatura pypdf textual rich fastapi uvicorn jinja2 python-multipart sse-starlette mcp pyyaml pytest

import sys
sys.path.insert(0, '/content/engine-repo')

In [ ]:
# Sanity: every module imports cleanly.
import engine, engine.core, engine.core.pipeline, engine.core.memory, engine.core.domains
print('engine version:', engine.__version__)
print('pipeline entry:', engine.core.build_graph)

## Step 2 — mock the LLM

We replace `engine.core.models.OpenAI` with a `MagicMock` that returns canned responses keyed by prompt fragments. This is the same technique the engine's own test suite uses (`engine/tests/test_interfaces.py::patched_graph`).

In [ ]:
from types import SimpleNamespace
from unittest import mock
import engine.core.models as _models

def _resp(text):
    return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=text))])

def canned_chat(*args, **kwargs):
    prompt = kwargs.get('messages', [{}])[0].get('content', '')
    if 'Classify this research question' in prompt:
        return _resp('multihop')
    if 'step-level verifier' in prompt:
        return _resp('VERDICT: accept\nFEEDBACK:')
    if 'Break this research question' in prompt:
        return _resp('sub-question one\nsub-question two\nsub-question three')
    if 'concise factual paragraph' in prompt:
        return _resp('Hypothetical answer text.')
    if 'Summarize these sources' in prompt:
        return _resp('Summary of sources with [1] and [2] citations.')
    if 'Compress each numbered chunk' in prompt:
        return _resp('[1] compressed one\n[2] compressed two\n[3] compressed three')
    if 'Answer the question using ONLY the evidence' in prompt:
        return _resp('Hybrid retrieval combines BM25 + dense + RRF [1]. It outperforms single-stage [2].')
    if 'List each standalone factual claim' in prompt:
        return _resp('CLAIM: Hybrid combines BM25 + dense + RRF\nVERIFIED: yes\nCLAIM: It outperforms single-stage\nVERIFIED: yes')
    return _resp('(unknown prompt — mocked fallback)')

client = mock.MagicMock()
client.chat.completions.create.side_effect = canned_chat
_models.OpenAI = mock.MagicMock(return_value=client)

# Also mock SearXNG so no HTTP request is made.
import engine.core.pipeline as _pipeline
import requests
def fake_get(url, params=None, timeout=None):
    r = mock.MagicMock()
    r.status_code = 200
    r.raise_for_status = mock.MagicMock()
    r.json = lambda: {'results': [
        {'url': 'https://example.com/a', 'title': 'Example A', 'content': 'Example A snippet about hybrid retrieval.'},
        {'url': 'https://example.com/b', 'title': 'Example B', 'content': 'Example B snippet on BM25 and dense fusion.'},
    ]}
    return r
requests.get = fake_get

# And short-circuit the embedder in core.rag so no embeddings call either.
from core.rag import HybridRetriever
HybridRetriever.__init__.__defaults__ = ()
def fake_embed(batch):
    return [[float(len(s)), float(len(s.split()))] for s in batch]
original_init = HybridRetriever.__init__
def patched_init(self, *a, **kw):
    original_init(self, *a, **kw)
    self.embedder = fake_embed
HybridRetriever.__init__ = patched_init

# Keep the pipeline batched (not streaming — our mock doesn't speak that protocol).
_pipeline.ENABLE_STREAM = False
# Fetch is network-bound — skip it; evidence stays as the mocked SearXNG snippets.
_pipeline.ENABLE_FETCH = False

print('LLM + SearXNG + embedder all mocked. Network-free.')

## Step 3 — invoke the graph directly

The pipeline builds a LangGraph and `.invoke()`s with an initial `State`. Watch how each node adds to the state.

In [ ]:
from engine.core import build_graph

graph = build_graph()
result = graph.invoke({
    'question': 'What is hybrid retrieval and how does it compare to dense-only retrieval?',
    'iterations': 0,
    'plan_rejects': 0,
    'trace': [],
})

print('question_class:', result['question_class'])
print()
print('subqueries:')
for s in result['subqueries']:
    print('  -', s[:80])
print()
print('evidence items:', len(result.get('evidence', [])))
print('answer:', result['answer'])
print()
print('verified/total claims:', sum(1 for c in result.get('claims', []) if c['verified']), '/', len(result.get('claims', [])))

## Step 4 — use `run_query` + `RunResult`

`engine.interfaces.common.run_query()` is the high-level wrapper the CLI, TUI, Web GUI, and MCP server all call. It handles memory injection and domain-preset application for you.

In [ ]:
from engine.interfaces.common import run_query, format_sources, format_trace_per_node, format_verified_summary

rr = run_query('Why does hybrid retrieval beat dense-only?', domain='general')

print('=== ANSWER ===')
print(rr.answer)
print()
print('=== HALLUCINATION CHECK ===')
print(format_verified_summary(rr))
for c in rr.verified_claims:
    print('  ✓', c['text'])
for c in rr.unverified_claims:
    print('  ✗', c)
print()
print('=== SOURCES ===')
for row in format_sources(rr):
    print(f"  [{row['idx']}] {row['url']}")
print()
print('=== TRACE (per-node totals) ===')
for row in format_trace_per_node(rr):
    print(f"  {row['node']:12s} calls={row['calls']:2d} latency={row['latency_s']:.2f}s tokens~{row['tokens_est']}")

## Step 5 — domain preset in action

Switch from `general` to `medical`. The preset appends domain-specific rules to the synthesize prompt.

In [ ]:
import engine.core.domains as _domains

for name in _domains.list_names():
    p = _domains.load(name)
    extra = (p.synthesize_prompt_extra or '').strip()
    print(f'--- {name} ---')
    print('  min_verified_ratio:', p.min_verified_ratio)
    print('  top_k_evidence:', p.top_k_evidence)
    print('  seed_queries:', p.seed_queries[:2], '…' if len(p.seed_queries) > 2 else '')
    print('  prompt_extra (first 80 chars):', (extra[:80] + '…') if len(extra) > 80 else extra)
    print()

In [ ]:
# Run the same question through a medical preset. Observe how the question gets augmented
# before the graph sees it — the synthesize node now operates under medical constraints.
rr_med = run_query('What is hybrid retrieval?', domain='medical')
print('answer:', rr_med.answer)
print('domain in result:', rr_med.domain)

## Next steps

- **Tutorial 02** — swap the mock for a real Groq API call.
- **Tutorial 03** — upload PDFs and query them.
- **Tutorial 04** — drive the engine from an MCP client.
- **Tutorial 05** — compare 6 domain presets side-by-side on the same question.

Docs: [`engine/README.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/engine/README.md), [`docs/architecture.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/docs/architecture.md).